# 01.7 — EDA Synthesis Report

Tổng hợp các tham số phát hiện từ các phân tích EDA thành báo cáo có cấu trúc JSON phục vụ các giai đoạn tiếp theo.


In [1]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np

# Reconfigure encoding
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')

# Path resolution
ROOT_DIR = Path.cwd()
while not (ROOT_DIR / "src").exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.eda.pipeline import EDAReport

print("=" * 70)
print("📑 02.7: SYNTHESIZING STANDALONE DOMAIN ARTIFACTS INTO EDA REPORT")
print("=" * 70)

INPUT_DIR = ROOT_DIR / "notebooks" / "eda" / "outputs"

# 1. Đọc 6 artifact domain
with open(INPUT_DIR / "threshold_report.json", "r", encoding="utf-8") as f:
    th_rep = json.load(f)
with open(INPUT_DIR / "distribution_report.json", "r", encoding="utf-8") as f:
    dist_rep = json.load(f)
with open(INPUT_DIR / "correlation_report.json", "r", encoding="utf-8") as f:
    corr_rep = json.load(f)
with open(INPUT_DIR / "temporal_structure_report.json", "r", encoding="utf-8") as f:
    temp_rep = json.load(f)
with open(INPUT_DIR / "stationarity_report.json", "r", encoding="utf-8") as f:
    stat_rep = json.load(f)
with open(INPUT_DIR / "extreme_events_report.json", "r", encoding="utf-8") as f:
    ext_rep = json.load(f)

print("✅ Đã nạp thành công cả 6 domain artifact JSON.")


📑 02.7: SYNTHESIZING STANDALONE DOMAIN ARTIFACTS INTO EDA REPORT
✅ Đã nạp thành công cả 6 domain artifact JSON.


In [2]:
# 2. Xây dựng EDAReport hợp nhất
report = EDAReport(
    suggested_sarima_order=tuple(stat_rep.get('suggested_sarima_order', [3, 0, 3])),
    suggested_seasonal_order=tuple(stat_rep.get('suggested_seasonal_order', [1, 1, 1, 7])),
    vif_filtered_features=corr_rep.get('vif_filtered_features', []),
    validated_rain_threshold=float(th_rep.get('chosen_threshold_mm', 0.1)),
    wet_season_months=[int(m) for m in temp_rep.get('wet_season_months', [5, 6, 7, 8, 9, 10, 11])],
    representative_periods=[int(p) for p in temp_rep.get('candidate_domain_periods', [7, 30, 122, 365])],
    high_skew_features=dist_rep.get('high_skew_features', []),
    extreme_event_thresholds={k: float(v) for k, v in ext_rep.get('thresholds', {}).items()},
    intermittency_adi=float(dist_rep.get('intermittency', {}).get('adi', 1.64)),
    intermittency_cv2=float(dist_rep.get('intermittency', {}).get('cv2', 2.02)),
    diagnostic_labels_optional={
        "intermittency_classification": dist_rep.get('intermittency', {}).get('classification', 'Lumpy'),
        "stationarity_adf": "Stationary",
        "wavelet_status": "excluded_from_primary_pipeline",
        "sarima_order_status": "EDA-suggested configuration"
    },
    raw_results={
        "threshold": th_rep,
        "distribution": dist_rep,
        "correlation": corr_rep,
        "temporal": temp_rep,
        "stationarity": stat_rep,
        "extreme": ext_rep
    }
)

# 3. Xuất ra file canonical data/processed/eda_report.json
output_file = ROOT_DIR / "data" / "processed" / "eda_report.json"
output_file.parent.mkdir(parents=True, exist_ok=True)
report.to_json(output_file, include_diagnostics=True)

# Đồng thời lưu vào notebooks/eda/outputs/eda_report.json
report.to_json(INPUT_DIR / "eda_report.json", include_diagnostics=True)

print(f"\n💾 Đã lưu cấu hình EDAReport thành công tại: {output_file}")



💾 Đã lưu cấu hình EDAReport thành công tại: D:\DS108_project\data\processed\eda_report.json


In [3]:
# 4. Xác thực Vòng tròn (Round-trip Verification)
loaded_report = EDAReport.from_json(output_file)

# Assertions downstream contract
assert loaded_report.validated_rain_threshold == 0.1, "Threshold mismatch!"
assert loaded_report.suggested_sarima_order == (3, 0, 3) or len(loaded_report.suggested_sarima_order) == 3
assert loaded_report.suggested_seasonal_order == (1, 1, 1, 7)
assert len(loaded_report.vif_filtered_features) > 0, "VIF features list is empty!"
assert loaded_report.wet_season_months == temp_rep.get('wet_season_months'), "Wet season mismatch!"
assert loaded_report.intermittency_adi > 1.32, "ADI should indicate intermittent series"
assert loaded_report.intermittency_cv2 > 0.49, "CV2 should indicate high variance"

print("✅ ROUND-TRIP VERIFICATION PASSED:")
print(f"   • validated_rain_threshold:    {loaded_report.validated_rain_threshold} mm")
print(f"   • suggested_sarima_order:      {loaded_report.suggested_sarima_order}")
print(f"   • suggested_seasonal_order:    {loaded_report.suggested_seasonal_order}")
print(f"   • vif_filtered_features count: {len(loaded_report.vif_filtered_features)}")
print(f"   • wet_season_months:           {loaded_report.wet_season_months}")
print(f"   • Intermittency (ADI, CV2):    ADI={loaded_report.intermittency_adi:.2f}, CV2={loaded_report.intermittency_cv2:.2f}")
print(f"   • Extreme Thresholds:          {loaded_report.extreme_event_thresholds}")
print("\n🎯 Pipeline Handshake: Sẵn sàng cung cấp tham số cho Feature Engineering & Modeling!")


✅ ROUND-TRIP VERIFICATION PASSED:
   • validated_rain_threshold:    0.1 mm
   • suggested_sarima_order:      (3, 0, 3)
   • suggested_seasonal_order:    (1, 1, 1, 7)
   • vif_filtered_features count: 3
   • wet_season_months:           [6, 7, 10]
   • Intermittency (ADI, CV2):    ADI=1.65, CV2=1.99
   • Extreme Thresholds:          {'p95': 17.19, 'p99': 34.16, 'p99_9': 61.59, 'max_daily_mm': 107.9}

🎯 Pipeline Handshake: Sẵn sàng cung cấp tham số cho Feature Engineering & Modeling!
